In [136]:
import numpy as np
import scipy as sp
import scipy.optimize as so
import matplotlib.pyplot as plt
%matplotlib inline

In [209]:
# solve min 1/2 x^T A x + b^T x
# subject to x>=0

def LCP_APGD(A,b,tau,Nmax,x0):
    xk=x0
    xhatk=np.ones(np.size(x0))
    yk=xk
    thetak=1
    xkdiff=xk-xhatk
    temp=A.dot(xkdiff)
    Lk=np.sqrt(temp.dot(temp)/xkdiff.dot(xkdiff))
    tk=1.0/Lk
    # first step
    ite=0
    for i in range(0,Nmax):
        ite+=1
        g=A.dot(yk)+b
        # better estimate of tk
        print("ite: ",ite)
        while True:
            xkp1=(yk-tk*g).clip(min=0) # projection into the feasible region x>=0
            xdiff=xkp1-yk
            if xkp1.dot(A.dot(xkp1))*0.5+xkp1.dot(b) - ( yk.dot(A.dot(yk))*0.5+yk.dot(b)+g.dot(xdiff)+0.5*Lk*(xdiff).dot(xdiff) ) < 0:
                break
            Lk*=2
            tk=1/Lk
            print("tk: ",tk)


        thetakp1 = (-thetak**2 + thetak*np.sqrt(4 + thetak**2))/2.
        betakp1=thetak*(1-thetak)/(thetak*thetak+thetakp1)
        ykp1=xkp1+betakp1*(xkp1-xk)
        resx=xkp1-xk
        print("xkp1: ",xkp1)
        print("ykp1: ",ykp1)
        if np.max(np.abs(resx)) < tau: # terminate loop
            break
        if g.dot(xkp1-xk)>0: # reset Nesterov parameters
            ykp1=xkp1
            thetakp1=1
        xk=xkp1
        yk=ykp1
        thetak=thetakp1
        Lk*=0.5 # 0.5 is a tunable parameter. IN the paper they used 0.9
        tk=1/Lk
    return xkp1,resx,ite

In [306]:
matDim=2000
Aroot=np.random.randn(matDim,matDim)+np.sqrt(matDim)*np.identity(matDim) # let cond(A) ~ O(1)
Amat=np.transpose(Aroot).dot(Aroot)
bvec=np.random.randn(matDim)
#print(Amat,bvec)
print(np.linalg.cond(Amat))

197110.979372


In [307]:
restarget=1e-5

xsol,resid,ite=LCP_APGD(Amat,bvec,restarget,10000,np.zeros(np.size(bvec)))
print(xsol,resid,ite)

ite:  1
xkp1:  [  0.00000000e+00   0.00000000e+00   9.61101704e-05 ...,   1.27450550e-04
   0.00000000e+00   2.38619182e-04]
ykp1:  [  0.00000000e+00   0.00000000e+00   9.61101704e-05 ...,   1.27450550e-04
   0.00000000e+00   2.38619182e-04]
ite:  2
tk:  0.000190848271521
xkp1:  [  0.00000000e+00   0.00000000e+00   8.70503290e-05 ...,   3.04552661e-04
   0.00000000e+00   2.59223394e-04]
ykp1:  [  0.00000000e+00   0.00000000e+00   8.44976868e-05 ...,   3.54451806e-04
   0.00000000e+00   2.65028703e-04]
ite:  3
tk:  0.000190848271521
xkp1:  [  0.00000000e+00   0.00000000e+00   7.56393013e-05 ...,   3.38181628e-04
   0.00000000e+00   4.15784820e-04]
ykp1:  [  0.00000000e+00   0.00000000e+00   7.06864271e-05 ...,   3.52778038e-04
   0.00000000e+00   4.83739178e-04]
ite:  4
tk:  0.000190848271521
tk:  9.54241357606e-05
xkp1:  [  0.00000000e+00   0.00000000e+00   6.82903710e-05 ...,   3.68619551e-04
   0.00000000e+00   4.58934945e-04]
ykp1:  [  0.00000000e+00   0.00000000e+00   6.43876200e-0

In [308]:
y=Amat.dot(xsol)+bvec
print(y)
print(xsol)
for i in range(len(xsol)):
    if xsol[i]<- restarget:
        print("res.x error: ", xsol[i])
    if y[i]< - restarget:
        print("y error: ", y[i])
    print(y[i]*xsol[i])

[  2.13748439e+00   1.31072274e+00   4.43689686e-03 ...,  -1.62675852e-03
   1.88397769e+00   1.03065215e-02]
[  0.00000000e+00   0.00000000e+00   7.12132547e-05 ...,   2.94078655e-04
   0.00000000e+00   5.91708064e-04]
0.0
0.0
3.15965866115e-07
3.19866324228e-06
3.71142112939e-06
0.0
0.0
0.0
y error:  -0.001646586818
-9.83957422648e-07
0.0
1.2620946604e-05
0.0
4.96089060537e-07
0.0
2.10737496802e-06
0.0
0.0
y error:  -0.0295437667433
-3.5638321436e-07
2.05745352692e-07
0.0
9.08695218083e-07
0.0
0.0
0.0
0.0
y error:  -0.00210658495946
-6.95591511777e-07
y error:  -0.000932450251926
-2.26273714495e-07
0.0
0.0
0.0
0.0
1.82451330077e-05
0.0
1.71497219707e-07
0.0
0.0
0.0
y error:  -0.0066488280215
-1.07780712286e-06
0.0
1.04606414015e-05
2.15903781896e-06
0.0
0.0
0.0
0.0
2.15376529791e-05
0.0
0.0
0.0
0.0
y error:  -0.0024773038213
-1.34173342884e-06
0.0
0.0
0.0
0.0
0.0
0.0
3.48435024769e-06
1.17118309573e-08
2.09048279267e-06
5.03271934116e-07
0.0
0.0
0.0
1.90512631638e-05
0.0
0.0
3.524829

2.34646464003e-06
0.0
1.7090742474e-05
y error:  -0.00217494987594
-8.41615093813e-07
1.39670888906e-05
0.0
0.0
0.0
0.0
0.0
1.29496036955e-07
1.88766137539e-05
3.15469924764e-06
4.86635482736e-05
0.0
0.0
1.08789255179e-05
y error:  -0.00814039238898
-3.44442235053e-08
0.0
1.51430826753e-05
0.0
0.0
1.01472467998e-05
0.0
1.30610367796e-08
6.61604141532e-06
6.3341283108e-06
0.0
y error:  -0.00668343302817
-1.78471907123e-07
y error:  -0.00346316252387
-1.18323917805e-06
0.0
0.0
1.86894012169e-05
0.0
0.0
1.45158637375e-05
0.0
0.0
0.0
8.12144443725e-06
0.0
0.0
0.0
0.0
y error:  -0.00395184139472
-1.28824323375e-06
y error:  -0.00669743149209
-1.92445222683e-06
3.80659759843e-06
y error:  -0.00445129751167
-2.2337680985e-06
5.54879353382e-07
4.88103070887e-07
0.0
y error:  -0.0199270398882
-1.09538269378e-06
1.60207015302e-07
3.03128716831e-06
0.0
0.0
0.0
3.4189845562e-06
5.56193573207e-07
0.0
y error:  -0.00794064684104
-1.64611883147e-06
5.03053120201e-06
6.86703673841e-08
3.45187332252e-0

In [309]:
bnds = [(0,None) for i in range(matDim)]
fun= lambda x: 0.5*x.dot(Amat.dot(x))+bvec.dot(x)
funjac= lambda x: (Amat.dot(x))+bvec
#print(bnds)
res=so.minimize(fun, np.zeros(np.size(bvec)), method='L-BFGS-B', bounds=bnds,tol=restarget)
print(res)

[(0, None), (0, None), (0, None), (0, None), (0, None), (0, None), (0, None), (0, None), (0, None), (0, None), (0, None), (0, None), (0, None), (0, None), (0, None), (0, None), (0, None), (0, None), (0, None), (0, None), (0, None), (0, None), (0, None), (0, None), (0, None), (0, None), (0, None), (0, None), (0, None), (0, None), (0, None), (0, None), (0, None), (0, None), (0, None), (0, None), (0, None), (0, None), (0, None), (0, None), (0, None), (0, None), (0, None), (0, None), (0, None), (0, None), (0, None), (0, None), (0, None), (0, None), (0, None), (0, None), (0, None), (0, None), (0, None), (0, None), (0, None), (0, None), (0, None), (0, None), (0, None), (0, None), (0, None), (0, None), (0, None), (0, None), (0, None), (0, None), (0, None), (0, None), (0, None), (0, None), (0, None), (0, None), (0, None), (0, None), (0, None), (0, None), (0, None), (0, None), (0, None), (0, None), (0, None), (0, None), (0, None), (0, None), (0, None), (0, None), (0, None), (0, None), (0, None)

      fun: -0.21625414513234392
 hess_inv: <2000x2000 LbfgsInvHessProduct with dtype=float64>
      jac: array([ 2.11386839,  1.39124016, -0.10635265, ...,  0.01619606,
        1.95125067,  0.0416677 ])
  message: b'STOP: TOTAL NO. of f AND g EVALUATIONS EXCEEDS LIMIT'
     nfev: 16008
      nit: 5
   status: 1
  success: False
        x: array([  0.00000000e+00,   0.00000000e+00,   1.93301706e-05, ...,
         3.51625897e-04,   0.00000000e+00,   6.03263806e-04])


In [310]:
y=Amat.dot(res.x)+bvec
print(y)
print(res.x)
for i in range(len(res.x)):
    if res.x[i]<-1e-5:
        print("res.x error")
    if y[i]<-1e-5:
        print("y error")
    print(y[i]*res.x[i])

[ 2.11384825  1.39122015 -0.10637244 ...,  0.01617573  1.95123048
  0.04164733]
[  0.00000000e+00   0.00000000e+00   1.93301706e-05 ...,   3.51625897e-04
   0.00000000e+00   6.03263806e-04]
0.0
0.0
y error
-2.05619749166e-06
y error
-1.64640288326e-05
5.30952945125e-05
0.0
0.0
0.0
2.49892565927e-05
0.0
y error
-2.93324017308e-05
0.0
2.97753069449e-06
0.0
4.1605983904e-06
0.0
0.0
7.3774639286e-06
8.08428161717e-06
0.0
y error
-2.96399930709e-06
0.0
0.0
0.0
0.0
y error
-5.47883003815e-07
5.75037553939e-06
3.23999966789e-06
0.0
0.0
0.0
1.11018973856e-05
0.0
y error
-2.75818904177e-06
0.0
0.0
0.0
y error
-5.18196985278e-06
0.0
y error
-1.40205494115e-05
y error
-3.35063833269e-05
0.0
0.0
0.0
0.0
4.1602141129e-05
0.0
0.0
0.0
0.0
2.31133674272e-05
9.91530838257e-07
0.0
0.0
6.93064301425e-08
0.0
0.0
y error
-3.41580340546e-05
y error
-3.13008821263e-08
y error
-4.29178351056e-05
5.27735432293e-05
0.0
0.0
0.0
0.000107640746104
0.0
0.0
y error
-2.70408243461e-05
9.29008834627e-06
y error
-1.256

y error
-8.2739994887e-07
0.0
0.0
0.0
4.5606661359e-05
5.84130722934e-05
0.0
y error
-2.02150642414e-05
y error
-1.71178493989e-05
y error
-1.31173863186e-05
0.0
0.0
0.0
y error
-9.90044978634e-08
0.0
y error
-0.0
0.000146181289257
3.581476993e-05
4.35668186734e-05
0.0
0.0
1.82956352174e-05
y error
-9.79702349418e-08
0.0
5.20612778507e-05
0.0
0.0
4.56813860412e-05
0.0
y error
-4.56756100114e-07
y error
-2.97148598915e-05
4.55818324021e-07
0.0
y error
-3.46519523664e-08
y error
-1.06148298723e-05
0.0
0.0
5.37820677027e-06
0.0
0.0
y error
-2.00520564915e-05
0.0
0.0
0.0
4.66856015414e-07
0.0
0.0
0.0
0.0
y error
-2.57841073671e-05
y error
-1.46093701894e-05
2.39750073889e-05
y error
-2.50056696016e-05
y error
-6.49046674596e-06
y error
-4.83676522779e-06
0.0
y error
-2.489738566e-06
y error
-2.29839228565e-06
8.22084952896e-06
0.0
0.0
0.0
y error
-1.82190958503e-05
y error
-3.46541794403e-05
0.0
y error
-1.23967799359e-05
2.09947395016e-05
0.0
1.59680689873e-06
0.0
1.1446306433e-06
0.0
y e

In [311]:
print(res.x)
print(xsol)
print(res.x-xsol)

[  0.00000000e+00   0.00000000e+00   1.93301706e-05 ...,   3.51625897e-04
   0.00000000e+00   6.03263806e-04]
[  0.00000000e+00   0.00000000e+00   7.12132547e-05 ...,   2.94078655e-04
   0.00000000e+00   5.91708064e-04]
[  0.00000000e+00   0.00000000e+00  -5.18830841e-05 ...,   5.75472413e-05
   0.00000000e+00   1.15557426e-05]


In [312]:
def AGD(A,b,tau,Nmax,x0):
    xk=x0
    xhatk=np.ones(np.size(x0))
    yk=xk
    thetak=1
    xkdiff=xk-xhatk
    temp=A.dot(xkdiff)
    Lk=np.sqrt(temp.dot(temp)/xkdiff.dot(xkdiff))
    tk=1.0/Lk
    # first step
    ite=0
    for i in range(0,Nmax):
        ite+=1
        g=A.dot(yk)+b
        # better estimate of tk
        print("ite: ",ite)
        while True:
            xkp1=yk-tk*g
            xdiff=xkp1-yk
            if xkp1.dot(A.dot(xkp1))*0.5+xkp1.dot(b) - ( yk.dot(A.dot(yk))*0.5+yk.dot(b)+g.dot(xdiff)+0.5*Lk*(xdiff).dot(xdiff) ) < 0:
                break
            Lk*=2
            tk=1/Lk
            print("tk: ",tk)

        thetakp1 = (-thetak**2 + thetak*np.sqrt(4 + thetak**2))/2.
        betakp1=thetak*(1-thetak)/(thetak*thetak+thetakp1)
        ykp1=xkp1+betakp1*(xkp1-xk)
        resx=xkp1-xk
        print("xkp1: ",xkp1)
        print("ykp1: ",ykp1)
        if np.max(np.abs(resx)) < tau: # terminate loop
            break
        if g.dot(xkp1-xk)>0: # reset Nesterov parameters
            ykp1=xkp1
            thetakp1=1
        xk=xkp1
        yk=ykp1
        thetak=thetakp1
        Lk*=0.7
        tk=1/Lk
    return xkp1,resx,ite

In [313]:
xsol,resid,ite=AGD(Amat,bvec,restarget,5000,np.zeros(np.size(bvec)))
print(xsol,resid,ite)

ite:  1
xkp1:  [ -3.07789383e-04  -4.01961374e-04   9.61101704e-05 ...,   1.27450550e-04
  -2.60892782e-04   2.38619182e-04]
ykp1:  [ -3.07789383e-04  -4.01961374e-04   9.61101704e-05 ...,   1.27450550e-04
  -2.60892782e-04   2.38619182e-04]
ite:  2
tk:  0.000136320193944
xkp1:  [-0.00043339 -0.00047516  0.00016221 ...,  0.0003693  -0.000275    0.00023773]
ykp1:  [-0.00046878 -0.00049578  0.00018084 ...,  0.00043745 -0.00027897
  0.00023748]
ite:  3
xkp1:  [-0.00052091 -0.00067448  0.00027067 ...,  0.00052413 -0.00057878
  0.00042919]
ykp1:  [-0.00055889 -0.000761    0.00031774 ...,  0.00059133 -0.00071064
  0.00051229]
ite:  4
tk:  0.000139102238718
tk:  6.95511193591e-05
xkp1:  [-0.00064177 -0.00081703  0.00032581 ...,  0.00068118 -0.00066435
  0.00049214]
ykp1:  [-0.00070596 -0.00089274  0.0003551  ...,  0.00076458 -0.0007098
  0.00052557]
ite:  5
xkp1:  [-0.00071565 -0.00095872  0.00040958 ...,  0.00080031 -0.00080882
  0.00053973]
ykp1:  [-0.00075989 -0.00104355  0.00045973 ...,  

  0.00244226]
ykp1:  [-0.01108939 -0.02470411  0.01492879 ...,  0.01332022 -0.03063972
  0.00260221]
ite:  80
tk:  7.49347481406e-05
xkp1:  [-0.01105166 -0.02469084  0.01494121 ...,  0.0133181  -0.0306472
  0.00258445]
ykp1:  [-0.01115354 -0.02502617  0.01503464 ...,  0.01345851 -0.03120816
  0.00272155]
ite:  81
xkp1:  [-0.01117815 -0.02504731  0.01503028 ...,  0.01346195 -0.03123248
  0.00274479]
ykp1:  [-0.01130015 -0.02539116  0.01511619 ...,  0.01360072 -0.03179703
  0.00289944]
ite:  82
xkp1:  [-0.01128154 -0.02539327  0.01512507 ...,  0.01360084 -0.03182172
  0.00289954]
ykp1:  [-0.01138131 -0.02572713  0.01521655 ...,  0.01373487 -0.03239034
  0.00304888]
ite:  83
tk:  0.000109234326736
xkp1:  [-0.01143194 -0.02576103  0.01520487 ...,  0.0137401  -0.03241897
  0.00308885]
ykp1:  [-0.01157714 -0.02611606  0.01528191 ...,  0.01387453 -0.03299556
  0.00327161]
ite:  84
tk:  7.80245190968e-05
xkp1:  [-0.01153756 -0.02610128  0.0152949  ...,  0.01387186 -0.03300231
  0.00325312]
ykp

xkp1:  [-0.0293504  -0.04802223  0.02171124 ...,  0.02322509 -0.076288    0.03696197]
ykp1:  [-0.02968802 -0.04826146  0.02182977 ...,  0.02343809 -0.0767889   0.0376411 ]
ite:  162
xkp1:  [-0.02971246 -0.04827294  0.02183048 ...,  0.02344951 -0.07679951
  0.03767693]
ykp1:  [-0.03006798 -0.04851913  0.02194756 ...,  0.02366988 -0.07730178
  0.03837896]
ite:  163
tk:  0.000122542324134
xkp1:  [-0.03005093 -0.04851169  0.02195778 ...,  0.02367646 -0.077304    0.03838041]
ykp1:  [-0.03038332 -0.04874614  0.02208279 ...,  0.02389934 -0.07779942
  0.03907126]
ite:  164
tk:  8.75302315246e-05
xkp1:  [-0.03042119 -0.04876359  0.02207585 ...,  0.0239064  -0.07780836
  0.03910743]
ykp1:  [-0.03078484 -0.049011    0.02219182 ...,  0.02413223 -0.07830372
  0.03982147]
ite:  165
tk:  6.25215939461e-05
xkp1:  [-0.03077314 -0.04900592  0.02219781 ...,  0.02413551 -0.07830424
  0.0398201 ]
ykp1:  [-0.03111885 -0.04924396  0.0223176  ...,  0.02436056 -0.07879131
  0.04052012]
ite:  166
xkp1:  [-0.031

tk:  6.87358161716e-05
xkp1:  [-0.06254275 -0.07319313  0.03133283 ...,  0.05497244 -0.10412257
  0.11226257]
ykp1:  [-0.06296595 -0.0735827   0.03143483 ...,  0.05549702 -0.10427728
  0.11330907]
ite:  244
xkp1:  [-0.06296884 -0.07358771  0.03143605 ...,  0.05550452 -0.10427533
  0.11332014]
ykp1:  [-0.06338978 -0.07397752  0.03153802 ...,  0.05603016 -0.10442625
  0.11436492]
ite:  245
xkp1:  [-0.06339686 -0.0739856   0.03153892 ...,  0.05604073 -0.10442358
  0.11438324]
ykp1:  [-0.06381972 -0.07437871  0.03164055 ...,  0.05657048 -0.10457004
  0.11543354]
ite:  246
xkp1:  [-0.06382837 -0.07438984  0.03164221 ...,  0.05658562 -0.1045661
  0.11545826]
ykp1:  [-0.06425472 -0.07478923  0.03174427 ...,  0.05712399 -0.10470691
  0.1165204 ]
ite:  247
xkp1:  [-0.06427609 -0.07480762  0.03174405 ...,  0.05714498 -0.10470157
  0.11656392]
ykp1:  [-0.06471845 -0.07522041  0.03184467 ...,  0.05769766 -0.10483542
  0.11765638]
ite:  248
tk:  0.000204485679097
tk:  0.000102242839548
xkp1:  [-0.0

  0.20638187]
ite:  324
tk:  0.000110156982394
xkp1:  [-0.10181752 -0.10916845  0.03908593 ...,  0.10227077 -0.1057022
  0.20638023]
ykp1:  [-0.10236426 -0.10961677  0.0391995  ...,  0.10284813 -0.10564376
  0.20760075]
ite:  325
tk:  7.86835588531e-05
xkp1:  [-0.10237977 -0.10962018  0.03919767 ...,  0.10284911 -0.1056423
  0.20762125]
ykp1:  [-0.1029369  -0.1100678   0.03930839 ...,  0.10342218 -0.10558294
  0.20885096]
ite:  326
xkp1:  [-0.10293628 -0.11007074  0.03931222 ...,  0.1034276  -0.10558295
  0.20885396]
ykp1:  [-0.10348773 -0.11051721  0.03942574 ...,  0.10400084 -0.10552414
  0.21007549]
ite:  327
xkp1:  [-0.10350887 -0.11052322  0.039425   ...,  0.10400459 -0.10552224
  0.21010505]
ykp1:  [-0.10407627 -0.1109716   0.03953676 ...,  0.10457635 -0.10546209
  0.21134481]
ite:  328
tk:  0.000114699065384
tk:  5.73495326918e-05
xkp1:  [-0.10406812 -0.11097246  0.03954094 ...,  0.10458048 -0.10546287
  0.21133723]
ykp1:  [-0.10462234 -0.11141764  0.03965584 ...,  0.10515118 -0

...

tk:  0.00010211771576
tk:  5.10588578799e-05
xkp1:  [-0.57094664 -0.57707812  0.11134916 ...,  0.40480795 -0.12937352
  0.57421576]
ykp1:  [-0.57149446 -0.57792331  0.11148408 ...,  0.40514706 -0.12937698
  0.57416873]
ite:  1038
xkp1:  [-0.57149509 -0.57792527  0.11148476 ...,  0.40514761 -0.12937723
  0.57416812]
ykp1:  [-0.57204197 -0.57876997  0.11161997 ...,  0.4054863  -0.12938093
  0.57412061]
ite:  1039
xkp1:  [-0.57204377 -0.578772    0.11162069 ...,  0.40548642 -0.12938066
  0.5741218 ]
ykp1:  [-0.57259087 -0.5796163   0.11175623 ...,  0.40582424 -0.12938409
  0.57407563]
ite:  1040
xkp1:  [-0.57259204 -0.57962037  0.11175766 ...,  0.40582544 -0.12938467
  0.57407415]
ykp1:  [-0.57313874 -0.5804663   0.11189423 ...,  0.4061635  -0.12938866
  0.57402664]
ite:  1041
tk:  0.000106328317118
xkp1:  [-0.57314112 -0.5804679   0.11189482 ...,  0.40616321 -0.12938802
  0.57402912]
ykp1:  [-0.57368863 -0.58131301  0.11203159 ...,  0.40650001 -0.12939137
  0.57398421]
ite:  1042
tk:  7.

xkp1:  [-0.61421028 -0.64572016  0.12401245 ...,  0.4306906  -0.13008956
  0.57114253]
ykp1:  [-0.61473899 -0.64657906  0.12418973 ...,  0.43099704 -0.13010374
  0.57111612]
ite:  1118
xkp1:  [-0.61473973 -0.64658064  0.12419035 ...,  0.43099737 -0.13010387
  0.57111625]
ykp1:  [-0.61526777 -0.64743882  0.12436778 ...,  0.43130332 -0.13011815
  0.57109005]
ite:  1119
xkp1:  [-0.61526928 -0.64744059  0.12436854 ...,  0.4313034  -0.13011795
  0.57109138]
ykp1:  [-0.6157974  -0.64829824  0.12454626 ...,  0.43160861 -0.130132    0.57106657]
ite:  1120
xkp1:  [-0.61579835 -0.648302    0.12454767 ...,  0.43160974 -0.13013269
  0.57106547]
ykp1:  [-0.61632601 -0.64916112  0.12472632 ...,  0.43191527 -0.13014739
  0.57103962]
ite:  1121
tk:  0.000119282275913
tk:  5.96411379567e-05
xkp1:  [-0.61632726 -0.6491615   0.12472657 ...,  0.43191489 -0.1301469
  0.57104153]
ykp1:  [-0.61685476 -0.65001872  0.12490498 ...,  0.43221923 -0.13016106
  0.57101766]
ite:  1122
xkp1:  [-0.61685554 -0.65002029

xkp1:  [-0.65432707 -0.71170642  0.13856834 ...,  0.45344401 -0.13115969
  0.57051576]
ykp1:  [-0.65482148 -0.71252757  0.13875688 ...,  0.45372173 -0.13116907
  0.57052554]
ite:  1196
xkp1:  [-0.65482211 -0.71252863  0.13875718 ...,  0.45372204 -0.13116889
  0.57052601]
ykp1:  [-0.65531592 -0.71334879  0.13894556 ...,  0.45399938 -0.13117806
  0.57053624]
ite:  1197
xkp1:  [-0.65531659 -0.71335049  0.13894605 ...,  0.454      -0.13117795
  0.57053641]
ykp1:  [-0.65580984 -0.71417029  0.13913444 ...,  0.45427727 -0.13118699
  0.57054677]
ite:  1198
xkp1:  [-0.65581139 -0.71417218  0.13913497 ...,  0.45427771 -0.13118641
  0.57054839]
ykp1:  [-0.65630496 -0.71499183  0.13932342 ...,  0.45455472 -0.13119485
  0.57056034]
ite:  1199
tk:  0.000131138124802
tk:  6.5569062401e-05
xkp1:  [-0.65630495 -0.714993    0.13932376 ...,  0.45455531 -0.13119503
  0.5705596 ]
ykp1:  [-0.65679728 -0.71581178  0.13951209 ...,  0.45483222 -0.13120364
  0.57057079]
ite:  1200
xkp1:  [-0.65679801 -0.7158127

xkp1:  [-0.69257903 -0.77554627  0.15315384 ...,  0.47510736 -0.1309597
  0.57266676]
ykp1:  [-0.69303762 -0.77631232  0.15332466 ...,  0.47536869 -0.1309451
  0.57270627]
ite:  1276
tk:  0.000100920655239
tk:  5.04603276196e-05
xkp1:  [-0.69303737 -0.77631295  0.15332484 ...,  0.47536909 -0.13094516
  0.57270552]
ykp1:  [-0.69349464 -0.77707784  0.15349544 ...,  0.4756302  -0.13093066
  0.57274418]
ite:  1277
xkp1:  [-0.69349484 -0.7770785   0.15349554 ...,  0.4756305  -0.13093052
  0.57274409]
ykp1:  [-0.69395124 -0.77784227  0.15366584 ...,  0.47589129 -0.13091592
  0.57278258]
ite:  1278
xkp1:  [-0.69395207 -0.777843    0.15366582 ...,  0.47589145 -0.13091551
  0.57278339]
ykp1:  [-0.69440823 -0.7786057   0.15383571 ...,  0.4761518  -0.13090052
  0.5728226 ]
ite:  1279
xkp1:  [-0.69440861 -0.77860705  0.15383591 ...,  0.4761524  -0.13090025
  0.57282238]
ykp1:  [-0.69486409 -0.77936932  0.1540056  ...,  0.47641273 -0.13088503
  0.57286128]
ite:  1280
tk:  0.000105081898416
xkp1:  [

ite:  1356
xkp1:  [-0.72787929 -0.83455516  0.16609961 ...,  0.49513372 -0.12901322
  0.57626421]
ykp1:  [-0.72829026 -0.83524132  0.16625003 ...,  0.49536225 -0.12898082
  0.57631299]
ite:  1357
tk:  8.08684282925e-05
xkp1:  [-0.72829015 -0.83524171  0.16625022 ...,  0.49536234 -0.12898075
  0.57631264]
ykp1:  [-0.72870011 -0.83592675  0.16640049 ...,  0.49559046 -0.12894834
  0.57636096]
ite:  1358
xkp1:  [-0.7287006  -0.83592706  0.16640056 ...,  0.49559028 -0.128948    0.57636156]
ykp1:  [-0.72911015 -0.8366109   0.16655058 ...,  0.49581773 -0.12891532
  0.57641038]
ite:  1359
xkp1:  [-0.72911005 -0.83661162  0.16655092 ...,  0.49581783 -0.12891513
  0.57640988]
ykp1:  [-0.72951859 -0.83729468  0.16670095 ...,  0.49604488 -0.12888233
  0.57645809]
ite:  1360
tk:  0.000117884006257
tk:  5.89420031286e-05
xkp1:  [-0.72951917 -0.83729471  0.16670089 ...,  0.49604463 -0.12888203
  0.57645896]
ykp1:  [-0.7299274  -0.83797629  0.16685054 ...,  0.49627094 -0.12884901
  0.57650793]
ite:  1

In [314]:
res=so.minimize(fun, np.zeros(np.size(bvec)),jac=funjac, method='Newton-CG',tol=restarget)
print(res)

res=so.minimize(fun, np.zeros(np.size(bvec)),method='BFGS',tol=restarget)
print(res)

     fun: -37.38281169805623
     jac: array([ -6.03781555e-05,  -6.57314990e-06,   5.39941052e-05, ...,
         7.99346034e-05,   6.36465450e-05,  -9.99949530e-05])
 message: 'Optimization terminated successfully.'
    nfev: 14
    nhev: 0
     nit: 13
    njev: 5250
  status: 0
 success: True
       x: array([-0.73675957, -0.85205314,  0.17054982, ...,  0.49521349,
       -0.12003844,  0.57879738])


KeyboardInterrupt: 

In [315]:
print(xsol-res.x)

[-0.00509599 -0.00583903  0.00071184 ...,  0.00756173 -0.00773752
 -0.00079913]


In [316]:
fun(xsol)

-37.361692467612102